In [10]:
import json
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.request import urlopen
import os
import time
from unidecode import unidecode
import warnings

In [11]:
# Put the match HTML file path here
match_html_path = r"C:\Users\K Raghunandan\Analyst\Matches\Newcastle 1-2 Arsenal - Premier League 2025_2026 Live.html"
# Put the Fotmob matchId here
fotmob_matchId = 4813432

def extract_json_from_html(html_path, save_output=False):
    with open(html_path, 'r', encoding='utf-8') as html_file:
        html = html_file.read()

    regex_pattern = r'(?<=require\.config\.params\["args"\].=.)[\s\S]*?;'
    data_txt = re.findall(regex_pattern, html)[0]

    # add quotations for JSON parser
    data_txt = data_txt.replace('matchId', '"matchId"')
    data_txt = data_txt.replace('matchCentreData', '"matchCentreData"')
    data_txt = data_txt.replace('matchCentreEventTypeJson', '"matchCentreEventTypeJson"')
    data_txt = data_txt.replace('formationIdNameMappings', '"formationIdNameMappings"')
    data_txt = data_txt.replace('};', '}')

    if save_output:
        # save JSON data to txt
        output_file = open(f"{html_path}.txt", "wt", encoding='utf-8')
        n = output_file.write(data_txt)
        output_file.close()

    return data_txt

def extract_data_from_dict(data):
    # load data from json
    event_types_json = data["matchCentreEventTypeJson"]
    formation_mappings = data["formationIdNameMappings"]
    events_dict = data["matchCentreData"]["events"]
    teams_dict = {data["matchCentreData"]['home']['teamId']: data["matchCentreData"]['home']['name'],
                  data["matchCentreData"]['away']['teamId']: data["matchCentreData"]['away']['name']}
    players_dict = data["matchCentreData"]["playerIdNameDictionary"]
    # create players dataframe
    players_home_df = pd.DataFrame(data["matchCentreData"]['home']['players'])
    players_home_df["teamId"] = data["matchCentreData"]['home']['teamId']
    players_away_df = pd.DataFrame(data["matchCentreData"]['away']['players'])
    players_away_df["teamId"] = data["matchCentreData"]['away']['teamId']
    players_df = pd.concat([players_home_df, players_away_df])
    players_ids = data["matchCentreData"]["playerIdNameDictionary"]
    return events_dict, players_df, teams_dict


json_data_txt = extract_json_from_html(match_html_path)
data = json.loads(json_data_txt)
events_dict, players_df, teams_dict = extract_data_from_dict(data)

df = pd.DataFrame(events_dict)
dfp = pd.DataFrame(players_df)

In [13]:
df.columns

Index(['id', 'eventId', 'minute', 'second', 'teamId', 'x', 'y',
       'expandedMinute', 'period', 'type', 'outcomeType', 'qualifiers',
       'satisfiedEventsTypes', 'isTouch', 'playerId', 'endX', 'endY',
       'relatedEventId', 'relatedPlayerId', 'blockedX', 'blockedY',
       'goalMouthZ', 'goalMouthY', 'isShot', 'isGoal', 'cardType'],
      dtype='object')

In [15]:
dfp.columns

Index(['playerId', 'shirtNo', 'name', 'position', 'height', 'weight', 'age',
       'isFirstEleven', 'isManOfTheMatch', 'field', 'stats',
       'subbedInPlayerId', 'subbedOutPeriod', 'subbedOutExpandedMinute',
       'subbedInPeriod', 'subbedInExpandedMinute', 'subbedOutPlayerId',
       'teamId'],
      dtype='object')

In [16]:
# Extract the 'displayName' value
df['type'] = df['type'].str.extract(r"'displayName': '([^']+)")
df['outcomeType'] = df['outcomeType'].str.extract(r"'displayName': '([^']+)")
df['period'] = df['period'].str.extract(r"'displayName': '([^']+)")

In [17]:
# New Column for Team Names and Oppositon TeamNames
df['teamName'] = df['teamId'].map(teams_dict)
team_names = list(teams_dict.values())
opposition_dict = {team_names[i]: team_names[1-i] for i in range(len(team_names))}
df['oppositionTeamName'] = df['teamName'].map(opposition_dict)

In [18]:
# Reshaping the data from 100x100 to 120x80, as I use the pitch_type='statsbomb', in the pitch function, you can consider according to your use
df['x'] = df['x']*1.2
df['y'] = df['y']*0.8
df['endX'] = df['endX']*1.2
df['endY'] = df['endY']*0.8
df['goalMouthY'] = df['goalMouthY']*0.8

In [20]:
columns_to_drop = ['height', 'weight', 'age', 'isManOfTheMatch', 'field', 'stats', 
                   'subbedInPlayerId', 'subbedOutPeriod', 
                   'subbedOutExpandedMinute', 'subbedInPeriod', 'subbedInExpandedMinute', 'subbedOutPlayerId', 
                   'teamId']
dfp.drop(columns=columns_to_drop, inplace=True)

In [21]:
# adding player name, shirt no. etc info
df = df.merge(dfp, on='playerId', how='left')

In [23]:
df.columns

Index(['id', 'eventId', 'minute', 'second', 'teamId', 'x', 'y',
       'expandedMinute', 'period', 'type', 'outcomeType', 'qualifiers',
       'satisfiedEventsTypes', 'isTouch', 'playerId', 'endX', 'endY',
       'relatedEventId', 'relatedPlayerId', 'blockedX', 'blockedY',
       'goalMouthZ', 'goalMouthY', 'isShot', 'isGoal', 'cardType', 'teamName',
       'oppositionTeamName', 'shirtNo', 'name', 'position', 'isFirstEleven',
       'shortName'],
      dtype='object')

In [24]:
df

,id,eventId,minute,second,teamId,x,y,expandedMinute,period,type,...,isShot,isGoal,cardType,teamName,oppositionTeamName,shirtNo,name,position,isFirstEleven,shortName
0,2.853077e+09,3,0,0.0,13,0.00,0.00,0,NaN,NaN,...,NaN,NaN,NaN,Arsenal,Newcastle,NaN,nan,NaN,NaN,nan
1,2.853077e+09,3,0,0.0,23,0.00,0.00,0,NaN,NaN,...,NaN,NaN,NaN,Newcastle,Arsenal,NaN,nan,NaN,NaN,nan
2,2.853077e+09,4,0,0.0,13,60.12,39.92,0,NaN,NaN,...,NaN,NaN,NaN,Arsenal,Newcastle,10.0,Eberechi Eze,AMC,True,E. Eze
3,2.853077e+09,5,0,5.0,13,34.44,45.60,0,NaN,NaN,...,NaN,NaN,NaN,Arsenal,Newcastle,1.0,David Raya,GK,True,D. Raya
4,2.853077e+09,4,0,7.0,23,32.88,24.08,0,NaN,NaN,...,NaN,NaN,NaN,Newcastle,Arsenal,4.0,Sven Botman,DC,True,S. Botman
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1459,2.853204e+09,910,98,58.0,13,0.00,0.00,106,NaN,NaN,...,NaN,NaN,NaN,Arsenal,Newcastle,NaN,nan,NaN,NaN,nan
1460,2.853206e+09,694,0,0.0,23,0.00,0.00,16,NaN,NaN,...,NaN,NaN,NaN,Newcastle,Arsenal,NaN,nan,NaN,NaN,nan
1461,2.853206e+09,913,0,0.0,13,0.00,0.00,16,NaN,NaN,...,NaN,NaN,NaN,Arsenal,Newcastle,NaN,nan,NaN,NaN,nan
1462,2.853004e+09,2,0,0.0,13,0.00,0.00,0,NaN,NaN,...,NaN,NaN,NaN,Arsenal,Newcastle,NaN,nan,NaN,NaN,nan


In [25]:
dfp

,playerId,shirtNo,name,position,isFirstEleven
0,105720,1,Nick Pope,GK,True
1,415174,21,Tino Livramento,DR,True
2,393355,12,Malick Thiaw,DC,True
3,379732,4,Sven Botman,DC,True
4,82277,33,Dan Burn,DL,True
5,201755,7,Joelinton,MC,True
6,338780,39,Bruno Guimarães,MC,True
7,343501,8,Sandro Tonali,MC,True
8,141486,23,Jacob Murphy,FWR,True
9,386606,27,Nick Woltemade,FW,True


In [37]:
df.columns

Index(['id', 'eventId', 'minute', 'second', 'teamId', 'x', 'y',
       'expandedMinute', 'period', 'type', 'outcomeType', 'qualifiers',
       'satisfiedEventsTypes', 'isTouch', 'playerId', 'endX', 'endY',
       'relatedEventId', 'relatedPlayerId', 'blockedX', 'blockedY',
       'goalMouthZ', 'goalMouthY', 'isShot', 'isGoal', 'cardType', 'teamName',
       'oppositionTeamName', 'shirtNo', 'name', 'position', 'isFirstEleven',
       'shortName'],
      dtype='object')

In [40]:
# Filter to keep only rows where x, y, endX, and endY all have values
df = df[
    (df['x'].notna()) & 
    (df['y'].notna()) & 
    (df['endX'].notna()) & 
    (df['endY'].notna())
]

In [41]:
df

,id,eventId,minute,second,teamId,x,y,expandedMinute,period,type,...,isShot,isGoal,cardType,teamName,oppositionTeamName,shirtNo,name,position,isFirstEleven,shortName
2,2.853077e+09,4,0,0.0,13,60.12,39.92,0,NaN,NaN,...,NaN,NaN,NaN,Arsenal,Newcastle,10.0,Eberechi Eze,AMC,True,E. Eze
3,2.853077e+09,5,0,5.0,13,34.44,45.60,0,NaN,NaN,...,NaN,NaN,NaN,Arsenal,Newcastle,1.0,David Raya,GK,True,D. Raya
7,2.853078e+09,5,0,13.0,23,6.36,18.08,0,NaN,NaN,...,NaN,NaN,NaN,Newcastle,Arsenal,12.0,Malick Thiaw,DC,True,M. Thiaw
8,2.853096e+09,128,0,16.0,13,49.44,73.92,0,NaN,NaN,...,NaN,NaN,NaN,Arsenal,Newcastle,33.0,Riccardo Calafiori,DL,True,R. Calafiori
9,2.853077e+09,7,0,20.0,13,33.36,58.40,0,NaN,NaN,...,NaN,NaN,NaN,Arsenal,Newcastle,6.0,Gabriel Magalhaes,DC,True,G. Magalhaes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1451,2.853203e+09,905,98,27.0,13,17.40,40.24,106,NaN,NaN,...,NaN,NaN,NaN,Arsenal,Newcastle,11.0,Gabriel Martinelli,Sub,NaN,G. Martinelli
1452,2.853203e+09,690,98,50.0,23,110.88,0.00,106,NaN,NaN,...,NaN,NaN,NaN,Newcastle,Arsenal,2.0,Kieran Trippier,Sub,NaN,K. Trippier
1455,2.853203e+09,692,98,52.0,23,111.24,26.80,106,NaN,NaN,...,NaN,NaN,NaN,Newcastle,Arsenal,33.0,Dan Burn,DL,True,D. Burn
1456,2.853203e+09,908,98,55.0,13,5.76,36.72,106,NaN,NaN,...,NaN,NaN,NaN,Arsenal,Newcastle,12.0,Jurrien Timber,DR,True,J. Timber


In [42]:
df.columns

Index(['id', 'eventId', 'minute', 'second', 'teamId', 'x', 'y',
       'expandedMinute', 'period', 'type', 'outcomeType', 'qualifiers',
       'satisfiedEventsTypes', 'isTouch', 'playerId', 'endX', 'endY',
       'relatedEventId', 'relatedPlayerId', 'blockedX', 'blockedY',
       'goalMouthZ', 'goalMouthY', 'isShot', 'isGoal', 'cardType', 'teamName',
       'oppositionTeamName', 'shirtNo', 'name', 'position', 'isFirstEleven',
       'shortName'],
      dtype='object')

In [44]:
df=df[["minute","second","x","y","endX","endY","teamName","shirtNo","position","shortName"]]

In [45]:
df

,minute,second,x,y,endX,endY,teamName,shirtNo,position,shortName
2,0,0.0,60.12,39.92,34.56,45.60,Arsenal,10.0,AMC,E. Eze
3,0,5.0,34.44,45.60,87.12,55.92,Arsenal,1.0,GK,D. Raya
7,0,13.0,6.36,18.08,57.36,3.12,Newcastle,12.0,DC,M. Thiaw
8,0,16.0,49.44,73.92,40.08,69.36,Arsenal,33.0,DL,R. Calafiori
9,0,20.0,33.36,58.40,20.04,27.20,Arsenal,6.0,DC,G. Magalhaes
...,...,...,...,...,...,...,...,...,...,...
1451,98,27.0,17.40,40.24,18.48,80.00,Arsenal,11.0,Sub,G. Martinelli
1452,98,50.0,110.88,0.00,111.24,26.80,Newcastle,2.0,Sub,K. Trippier
1455,98,52.0,111.24,26.80,111.00,39.04,Newcastle,33.0,DL,D. Burn
1456,98,55.0,5.76,36.72,38.52,44.96,Arsenal,12.0,DR,J. Timber
